# Equilibrium and transient optical response of the parabolic two-band model

Analysis of the parabolic two-band model to study the Dynamical Franz-Keldysh Effect (DFKE), adapted from the lecture notes of Shunsuke Sato. From time-dependent traces of vector potential, current and field, the notebook computes

1. the **equilibrium** optical conductivity $\sigma(\omega)$ from a probe-only run,
2. the **transient** conductivity $\tilde\sigma(\omega, T)$ at a single pump-probe delay,
3. the **transient absorption spectrum** $\Delta\sigma(\omega, T)$ over a scan of delays.

**Data.** The traces come from the 1D parabolic two-band simulator in `twoband.simulate` (see the README). If `data/` is empty, the first code cell generates them, which takes about 2 minutes. To regenerate by hand: `twoband-generate --out data`. These are model results, not measured data.

**Sign convention.** The simulator uses the physical sign of the current (electron charge $-1$), so absorption appears as **positive** Re[$\sigma$]. The original Fortran output used the opposite sign, which is why earlier versions of this analysis plotted $-$Re[$\sigma$].

In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

import twoband as tb

DATA_DIR = Path("data")
meta = tb.load_or_generate(DATA_DIR)
files = meta["files"]
cols = meta["columns"]
COL_TIME, COL_A, COL_J, COL_E = (
    cols["time"],
    cols["vector_potential"],
    cols["current"],
    cols["field"],
)
BLOCK_LENGTH = meta["block_length"]
DELAYS_FS = np.array(meta["config"]["delays_fs"])
print(meta["description"])
print(f"{len(DELAYS_FS)} delays from {DELAYS_FS[0]:.2f} to {DELAYS_FS[-1]:.2f} fs")

## Settings

In [ ]:
# Energy grid and damping, shared by ALL spectra so they can be compared directly
ENERGY = np.linspace(2.0, 12.0, 500) / tb.HARTREE_TO_EV  # Hartree
GAMMA = 0.1 / tb.HARTREE_TO_EV  # damping in the Fourier transform
ENERGY_EV = ENERGY * tb.HARTREE_TO_EV

## 1. Equilibrium optical conductivity

$$\sigma(\omega) = \frac{J(\omega)}{E(\omega)}, \qquad F(\omega) = \int F(t)\, e^{i\omega t - \gamma t}\, dt$$

In [ ]:
probe = np.loadtxt(DATA_DIR / files["probe"])
time = probe[:, COL_TIME]
probe_current = probe[:, COL_J]
probe_field = probe[:, COL_E]

fig, axs = plt.subplots(3, 1, figsize=(6, 7), sharex=True)
axs[0].plot(time * tb.AU_TO_FS, probe[:, COL_A])
axs[0].set_ylabel("A(t) (a.u.)")
axs[1].plot(time * tb.AU_TO_FS, probe_current, color="r")
axs[1].set_ylabel("J(t) (a.u.)")
axs[2].plot(time * tb.AU_TO_FS, probe_field)
axs[2].set_ylabel(r"E$_{probe}$(t) (a.u.)")
axs[2].set_xlabel("Time (fs)")
plt.show()

In [ ]:
current_w = tb.damped_ft(time, probe_current, ENERGY, GAMMA)
field_w = tb.damped_ft(time, probe_field, ENERGY, GAMMA)
sigma_eq = current_w / field_w

fig, axs = plt.subplots(2, 1, figsize=(6, 5), sharex=True)
axs[0].plot(ENERGY_EV, np.abs(field_w), color="dodgerblue", label="FT probe field")
axs[0].set_ylabel("Intensity (arb. units)")
axs[0].legend()
axs[1].plot(ENERGY_EV, np.abs(current_w), color="red", label="FT current")
axs[1].set_ylabel("Intensity (arb. units)")
axs[1].set_xlabel("Energy (eV)")
axs[1].legend()
plt.show()

plt.plot(ENERGY_EV, np.real(sigma_eq), color="red", linewidth=2, label="Equilibrium")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\sigma(\omega)$] (a.u.)")
plt.legend()
plt.show()

### Check against the analytic result

In the weak-probe limit the simulated conductivity must equal $\mathrm{Re}\,\sigma = \omega\,\mathrm{Im}\,\epsilon(\omega) / 4\pi$ of the analytic 1D model, broadened by the Lorentzian of width $\gamma$ that the damped transform introduces.

In [ ]:
cfg = meta["config"]
e_gap = cfg["e_gap_ev"] / tb.HARTREE_TO_EV
m_r = tb.reduced_mass(cfg["m_c"], cfg["m_v"])
analytic = tb.re_sigma_1d_broadened(ENERGY, e_gap, m_r, cfg["p_vc"], GAMMA)

plt.plot(ENERGY_EV, np.real(sigma_eq), lw=2, label="simulation")
plt.plot(ENERGY_EV, analytic, "k--", label="analytic")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\sigma(\omega)$] (a.u.)")
plt.legend()
plt.show()

## 2. Pump-only reference

In [ ]:
pump = np.loadtxt(DATA_DIR / files["pump"])
pump_time = pump[:, COL_TIME]
pump_current = pump[:, COL_J]
pump_field = pump[:, COL_E]

fig, axs = plt.subplots(3, 1, figsize=(6, 7), sharex=True)
axs[0].plot(pump_time * tb.AU_TO_FS, pump[:, COL_A])
axs[0].set_ylabel("A(t) (a.u.)")
axs[1].plot(pump_time * tb.AU_TO_FS, pump_current, color="r")
axs[1].set_ylabel(r"J$_{pump}$(t) (a.u.)")
axs[2].plot(pump_time * tb.AU_TO_FS, pump_field)
axs[2].set_ylabel(r"E$_{pump}$(t) (a.u.)")
axs[2].set_xlabel("Time (fs)")
plt.show()

## 3. Transient conductivity at a single delay

The probe response is isolated by subtracting the pump-only run:

$$J_{probe}(t) = J_{pump\text{-}probe}(t) - J_{pump}(t), \qquad E_{probe}(t) = E_{pump\text{-}probe}(t) - E_{pump}(t)$$

In [ ]:
pp = np.loadtxt(DATA_DIR / files["pump_probe"])
pp_time = pp[:, COL_TIME]
pp_probe_current = tb.subtract_reference(pp[:, COL_J], pump_current)
pp_probe_field = tb.subtract_reference(pp[:, COL_E], pump_field)
print(f"single delay: {meta['config']['single_delay_fs']} fs")

sigma_single = tb.optical_conductivity(pp_time, pp_probe_current, pp_probe_field, ENERGY, GAMMA)
delta_single = tb.transient_absorption(sigma_single, sigma_eq)[0]

fig, axs = plt.subplots(2, 1, figsize=(7, 7), sharex=True)
axs[0].plot(ENERGY_EV, np.real(sigma_single), label="Transient")
axs[0].plot(ENERGY_EV, np.real(sigma_eq), label="Equilibrium")
axs[0].set_ylabel(r"Re[$\sigma(\omega)$] (a.u.)")
axs[0].legend()
axs[1].plot(ENERGY_EV, np.real(delta_single))
axs[1].set_ylabel(r"Re[$\Delta\sigma(\omega)$] (a.u.)")
axs[1].set_xlabel("Energy (eV)")
plt.show()

## 4. Transient absorption over a delay scan

The scan file holds one block of `BLOCK_LENGTH` rows per delay. `split_delays` refuses data that does not divide evenly, so a truncated file is caught instead of silently producing a wrong map.

$$\Delta\sigma(\omega, T) = \tilde\sigma(\omega, T) - \sigma(\omega)$$

In [ ]:
scan = np.loadtxt(DATA_DIR / files["delay_scan"])
blocks = tb.split_delays(scan, BLOCK_LENGTH)  # (n_delays, BLOCK_LENGTH, n_columns)
scan_time = blocks[0, :, COL_TIME]

scan_probe_current = tb.subtract_reference(blocks[:, :, COL_J], pump_current)
scan_probe_field = tb.subtract_reference(blocks[:, :, COL_E], pump_field)
sigma_scan = tb.optical_conductivity(scan_time, scan_probe_current, scan_probe_field, ENERGY, GAMMA)
tas = tb.transient_absorption(sigma_scan, sigma_eq)  # (n_delays, n_energy)

plt.plot(ENERGY_EV, np.real(tas).T, linewidth=1)
plt.xlabel("Energy (eV)")
plt.ylabel(r"Re[$\Delta\sigma(\omega,T)$] (a.u.)")
plt.show()

### Transient absorption map

Bleaching at the band edge, induced absorption above and below it, and an oscillation with the delay at twice the pump frequency are the signatures of the Dynamical Franz-Keldysh effect.

In [ ]:
limit = np.abs(np.real(tas)).max()

fig, ax = plt.subplots(figsize=(5, 4))
mesh = ax.pcolormesh(
    DELAYS_FS,
    ENERGY_EV,
    np.real(tas).T,
    cmap="seismic",
    vmin=-limit,
    vmax=limit,
    shading="nearest",
)
fig.colorbar(mesh, ax=ax, label=r"Re[$\Delta\sigma$] (a.u.)")
ax.set_ylim(3, 8)
ax.set_xlabel("Pump-probe delay (fs)")
ax.set_ylabel("Energy (eV)")
plt.show()

## Reference

S. A. Sato et al., *Applied Sciences* 8(10), 1777 (2018). https://www.mdpi.com/2076-3417/8/10/1777